# MevzuatRadar — mT5 eğitimi (v2)

**v1'den farklar:** her şey Google Drive'a kaydedilir (oturum kapansa bile kaybolmaz), eğitim bittiğinde
hem geliştirme hem test tahminleri tek seferde üretilir, Drive'da eğitilmiş model varsa eğitim atlanır.

**Başlamadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.46" datasets sentencepiece accelerate

## 0. Google Drive'ı bağla
Açılan pencerede hesabını seç ve **tüm izin kutularını işaretleyip** devam et.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
KLASOR = "/content/drive/MyDrive/mevzuatradar"
MODEL_KLASORU = f"{KLASOR}/mt5_v2"
os.makedirs(KLASOR, exist_ok=True)
print("Çalışma klasörü:", KLASOR)

## 1. Veriyi yükle
`data/ml/` klasöründeki **üç** dosyayı birlikte seç: `seq2seq_train.jsonl`, `seq2seq_dev.jsonl`, `seq2seq_test.jsonl`.
Dosyalar Drive'a da kopyalanır; oturum kapanırsa tekrar yüklemen gerekmez.

In [ ]:
import shutil
from google.colab import files

GEREKLI = ["seq2seq_train.jsonl", "seq2seq_dev.jsonl", "seq2seq_test.jsonl"]
if not all(os.path.exists(f"{KLASOR}/{f}") for f in GEREKLI):
    yuklenen = files.upload()
    for ad in yuklenen:
        hedef = next((g for g in GEREKLI if ad.startswith(g.replace(".jsonl", ""))), None)
        if hedef:
            shutil.copy(ad, f"{KLASOR}/{hedef}")
eksik = [f for f in GEREKLI if not os.path.exists(f"{KLASOR}/{f}")]
print("Eksik dosya:", eksik or "yok, hepsi Drive'da")

In [ ]:
import json, random
import numpy as np
import torch

def oku(yol):
    return [json.loads(l) for l in open(yol, encoding="utf-8")]

train = oku(f"{KLASOR}/seq2seq_train.jsonl")
dev = oku(f"{KLASOR}/seq2seq_dev.jsonl")
test = oku(f"{KLASOR}/seq2seq_test.jsonl")
GUVENILIR = {"dogrulanmis", "kayitsiz"}
dev_guvenilir = [r for r in dev if r["quality"] in GUVENILIR]
print(f"eğitim {len(train)} | geliştirme {len(dev)} (güvenilir {len(dev_guvenilir)}) | test {len(test)}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED);

## 2. Model ve tokenizer
Eğitim hedefleri en fazla ~372 token; tahmin sırasında uzun çıktılara yer olsun diye üretim sınırı 512.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/mt5-small"
ONEK = "degisiklik: "
MAX_GIRDI, MAX_HEDEF, MAX_URETIM = 384, 400, 512

HAZIR = os.path.exists(f"{MODEL_KLASORU}/config.json")
kaynak = MODEL_KLASORU if HAZIR else MODEL
tokenizer = AutoTokenizer.from_pretrained(kaynak)
model = AutoModelForSeq2SeqLM.from_pretrained(kaynak)
print("Drive'daki eğitilmiş model yüklendi, eğitim atlanacak." if HAZIR else "Temel model yüklendi, eğitilecek.")

uzunluk = lambda xs: max(len(tokenizer(x).input_ids) for x in xs)
print("en uzun eğitim girdisi (token):", uzunluk([ONEK + r["input"] for r in train]))
print("en uzun eğitim hedefi (token):", uzunluk([r["target"] for r in train]))

## 3. Eğitim
Drive'da eğitilmiş model varsa bu hücre hiçbir şey yapmaz. Kontrol noktaları Drive'a yazılır.
~40-60 dakika sürer; sekmeyi açık tut.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def hazirla(orn):
    enc = tokenizer([ONEK + x for x in orn["input"]], max_length=MAX_GIRDI, truncation=True)
    enc["labels"] = tokenizer(text_target=orn["target"], max_length=MAX_HEDEF, truncation=True)["input_ids"]
    return enc

def normalize(s):
    return " ;; ".join(sorted(p.strip() for p in s.split(";;") if p.strip()))

def metrikler(tahmin):
    preds, labels = tahmin
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    p = tokenizer.batch_decode(preds, skip_special_tokens=True)
    g = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"tam_eslesme": float(np.mean([normalize(a) == normalize(b) for a, b in zip(p, g)]))}

if not HAZIR:
    sut = ["input", "target"]
    ds_train = Dataset.from_list([{k: r[k] for k in sut} for r in train]).map(hazirla, batched=True, remove_columns=sut)
    ds_dev = Dataset.from_list([{k: r[k] for k in sut} for r in dev_guvenilir]).map(hazirla, batched=True, remove_columns=sut)
    args = Seq2SeqTrainingArguments(
        output_dir=f"{KLASOR}/checkpoints_v2",
        num_train_epochs=40,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=1e-3,
        optim="adafactor",
        warmup_steps=64,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="tam_eslesme",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_HEDEF,
        generation_num_beams=1,
        logging_steps=10,
        report_to="none",
        seed=SEED,
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=ds_train, eval_dataset=ds_dev,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
        processing_class=tokenizer, compute_metrics=metrikler,
    )
    trainer.train()
    print("En iyi tam eşleşme:", trainer.state.best_metric)
    model = trainer.model
    model.save_pretrained(MODEL_KLASORU)
    tokenizer.save_pretrained(MODEL_KLASORU)
    print("Model Drive'a kaydedildi:", MODEL_KLASORU)

## 4. Geliştirme ve test setlerinin TAMAMINDA tahmin
Tahminler Drive'a yazılır ve bilgisayara indirilir.

In [ ]:
def tahmin_et(satirlar, dosya):
    model.eval()
    cihaz = model.device
    sonuc = []
    for i in range(0, len(satirlar), 8):
        grup = satirlar[i:i + 8]
        enc = tokenizer([ONEK + r["input"] for r in grup], max_length=MAX_GIRDI, truncation=True,
                        padding=True, return_tensors="pt").to(cihaz)
        with torch.no_grad():
            cikti = model.generate(**enc, max_new_tokens=MAX_URETIM, num_beams=4)
        for r, t in zip(grup, tokenizer.batch_decode(cikti, skip_special_tokens=True)):
            sonuc.append({"id": r["id"], "prediction": t})
    with open(f"{KLASOR}/{dosya}", "w", encoding="utf-8") as f:
        for p in sonuc:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"{dosya}: {len(sonuc)} tahmin -> {KLASOR}/{dosya}")
    return sonuc

if torch.cuda.is_available():
    model = model.to("cuda")
p_dev = tahmin_et(dev, "preds_dev_v2.jsonl")
p_test = tahmin_et(test, "preds_test_v2.jsonl")
for r, p in list(zip(test, p_test))[:4]:
    print("\nGİRDİ :", r["input"][:160])
    print("HEDEF :", r["target"][:200])
    print("MODEL :", p["prediction"][:200])

In [ ]:
files.download(f"{KLASOR}/preds_dev_v2.jsonl")
files.download(f"{KLASOR}/preds_test_v2.jsonl")

## 5. Sonra
İndirilen dosyaları projede `data/ml/` klasörüne koyup bilgisayarda çalıştır:
```
mevzuatradar evaluate-model --pred data/ml/preds_dev_v2.jsonl --split dev
mevzuatradar evaluate-model --pred data/ml/preds_test_v2.jsonl --split test
```